In [1]:
# Cell 1: Setup — imports, env config (.env), logging, OpenTelemetry
import sys, os, json, logging
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

from utils.logging_config import setup_logging, log_routing_decision, log_self_reflection
from utils.telemetry import setup_telemetry

# Structured JSON logger
logger = setup_logging(level=logging.INFO)

# OpenTelemetry tracer
tracer = setup_telemetry()

# Azure OpenAI config
AZURE_OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_DEPLOYMENT = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')
AZURE_OPENAI_AUTH_MODE = os.environ.get('AZURE_OPENAI_AUTH_MODE', 'entra').lower()
AZURE_OPENAI_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY')

if AZURE_OPENAI_AUTH_MODE not in {'entra', 'key'}:
    raise ValueError(f'Unsupported AZURE_OPENAI_AUTH_MODE: {AZURE_OPENAI_AUTH_MODE}')

def create_kernel() -> Kernel:
    kernel = Kernel()
    service_kwargs = {
        'service_id': 'azure_openai',
        'deployment_name': AZURE_OPENAI_DEPLOYMENT,
        'endpoint': AZURE_OPENAI_ENDPOINT,
        'api_version': AZURE_OPENAI_API_VERSION,
    }

    if AZURE_OPENAI_AUTH_MODE == 'entra':
        credential = DefaultAzureCredential()
        token_provider = get_bearer_token_provider(
            credential,
            'https://cognitiveservices.azure.com/.default',
        )
        service_kwargs['ad_token_provider'] = token_provider
    else:
        if not AZURE_OPENAI_API_KEY:
            raise ValueError('AZURE_OPENAI_API_KEY is required when AZURE_OPENAI_AUTH_MODE=key')
        service_kwargs['api_key'] = AZURE_OPENAI_API_KEY

    kernel.add_service(
        AzureChatCompletion(**service_kwargs)
    )
    return kernel

print(f'Setup complete')
print(f'  Endpoint: {AZURE_OPENAI_ENDPOINT[:50]}...')
print(f'  Deployment: {AZURE_OPENAI_DEPLOYMENT}')
print(f'  API version: {AZURE_OPENAI_API_VERSION}')
print(f'  Auth mode: {AZURE_OPENAI_AUTH_MODE}')
print(f'  SK version: {__import__("semantic_kernel").__version__}')

[Telemetry] ⚠️ azure-monitor-opentelemetry-exporter not installed. Using console exporter.
Setup complete
  Endpoint: https://dlstmvprtus-wingnut0310-ai.openai.azure.co...
  Deployment: gpt-4o
  API version: 2024-12-01-preview
  Auth mode: entra
  SK version: 1.41.2


In [3]:
# Cell 2: Define 3 dummy agents + load mock KB data
from agents.common.recommend_agent import RecommendPlugin
from agents.common.search_agent import SearchPlugin
from agents.common.policy_agent import PolicyPlugin

with open('mock_data/products.json', encoding='utf-8') as f:
    PRODUCTS = json.load(f)
with open('mock_data/search_index.json', encoding='utf-8') as f:
    SEARCH_INDEX = json.load(f)
with open('mock_data/policies.json', encoding='utf-8') as f:
    POLICIES = json.load(f)

rp = RecommendPlugin()
sp = SearchPlugin()
pp = PolicyPlugin()

print(f'Products catalog: {len(PRODUCTS)} items')
for p in PRODUCTS:
    stock = 'in stock' if p['in_stock'] else 'out of stock'
    print(f'  {p["name"]} ({p["brand"]}) - {p["price"]:,}won | {stock}')
print(f'\nSearch index: {len(SEARCH_INDEX)} entries')
print(f'Policy docs: {len(POLICIES)} documents')
print()
print('Demo queries:')
print('  1. 건성 피부에 좋은 크림 추천해줘')
print('  2. 설화수 자음생 크림 가격 알려줘')
print('  3. 교환 환불 정책 알려줘')
print('  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?')


Products catalog: 5 items
  설화수 자음생 크림 (설화수) - 120,000won | in stock
  라네즈 워터뱅크 하이드로 크림 (라네즈) - 42,000won | in stock
  이니스프리 그린티 씨드 세럼 (이니스프리) - 29,000won | in stock
  헤라 블랙 쿠션 파운데이션 (헤라) - 55,000won | in stock
  마몽드 로즈워터 토너 (마몽드) - 18,000won | out of stock

Search index: 5 entries
Policy docs: 3 documents

Demo queries:
  1. 건성 피부에 좋은 크림 추천해줘
  2. 설화수 자음생 크림 가격 알려줘
  3. 교환 환불 정책 알려줘
  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?


---
# SECTION 1a: BEFORE — Generic Black Box

**Single system-prompt completion. NO plugins, NO function calling.**

3개 역할을 system prompt에 직접 기술하고, 단일 LLM completion으로 응답합니다.  
어떤 역할이 응답을 처리했는지 **전혀 알 수 없는** 완전한 black box입니다.

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [4]:
# Cell 4: Section 1a — Run black-box supervisor
from agents.before.blackbox_supervisor import run_blackbox_supervisor

before_kernel = create_kernel()

queries_1a = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1a: Generic Black Box')
print('(NO plugins, NO function calling)')
print('=' * 60)

for q in queries_1a:
    print(f'\nQuery: {q}')
    response = await run_blackbox_supervisor(q, before_kernel)
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> No way to know which role handled this query')
    print('-' * 60)


SECTION 1a: Generic Black Box
(NO plugins, NO function calling)

Query: 건성 피부에 좋은 크림 추천해줘
Response:
건성 피부에는 수분 보충과 보습 유지에 효과적인 크림을 사용하는 것이 중요합니다. 다음과 같은 제품을 추천드립니다:

1. **아벤느 시칼파트 레스터레이티브 크림**  
   - 민감하고 건조한 피부를 진정시키는 데 도움을 줍니다.
   - 세라마이드 성분으로 보습력을 강화합니다.

2. **라로슈포제 시카플라스트 밤 B5**  
   - 판테놀과 셰어버터 성분이 건조한 피부를 촉촉하게 유지시켜 줍니다.
   - 장벽 강화 효과도 있습니다.

3. **달팡 하이드라스킨 리치 크림**  
   - 해조류 추출물과 보습 성분으로 피부를 깊게 촉촉하게 해줍니다.
   - 아주 건조한 피부에 적합합니다.

4. **유세린 울트라 센서티브 리페어 크림**  
   - 세라마이드와 자연유래 보습 성분들이 피부 장벽을 보호합니다.
   - 장시간 보습 효과가 있습니다.

피부 타입과 민감도를 고려해 테스트 후 사용해 보세요!

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------------

Query: 설화수 자음생 크림 가격 알려줘
Response:
설화수 자음생크림의 가격은 제품 종류 및 판매처에 따라 다를 수 있습니다. 일반적으로 75ml 기준 정가가 약 280,000원으로 책정되어 있습니다. 현재 정확한 가격과 재고 확인을 원하시면 가까운 설화수 매장이나 공식 온라인 스토어를 확인해보시길 권장합니다.

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------

---
# SECTION 1b: BEFORE — OK Architecture Mirror

**고객 현재 아키텍처 재현:** `ChatCompletionAgent` + plugins + `invoke()`

OK의 `ok_sk_agent_flow` 구조를 mirror합니다:
- Supervisor agent가 3개 plugin을 가지고 있음
- `invoke()` 한 번에 routing + tool calling + 응답 생성이 모두 처리됨
- **Routing decision이 LLM call 안에 buried** → 별도 로깅 hook 없음

```
ChatCompletionAgent(kernel, supervisor_instructions)
    └── invoke(history)
        ├── LLM decides which plugin/tool to call   ◄── ROUTING HERE (no hook)
        ├── tool calls execute                       ◄── NO SEPARATE LOG
        └── response streams back                    ◄── COMBINED OUTPUT
```

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [5]:
# Cell 6: Section 1b — Run OK pattern supervisor
from agents.before.blackbox_ok_pattern import run_ok_pattern_supervisor

ok_kernel = create_kernel()

queries_1b = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1b: OK Architecture Mirror')
print('(ChatCompletionAgent + Plugins)')
print('=' * 60)

for q in queries_1b:
    print(f'\nQuery: {q}')
    # <<< ROUTING DECISION HAPPENS HERE — buried inside LLM call, no hook available >>>
    response = await run_ok_pattern_supervisor(q, ok_kernel)
    # <<< TOOL SELECTION — no separate logging, combined with response generation >>>
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> Tool calls happened but no routing-level visibility')
    # <<< This is why the customer cannot see which agent was selected or why >>>
    print('-' * 60)


SECTION 1b: OK Architecture Mirror
(ChatCompletionAgent + Plugins)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool RecommendPlugin-recommend_product",
    "context": {
        "trace_id": "0x3d618fadfb0407104a5451510046621a",
        "span_id": "0xebd6e725d913d6b9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xa196573c5e335e15",
    "start_time": "2026-04-14T04:05:16.830252Z",
    "end_time": "2026-04-14T04:05:16.830511Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "RecommendPlugin-recommend_product",
        "gen_ai.tool.call.id": "call_u4fm2AHWpjk5WoRqtUuDKikY",
        "gen_ai.tool.description": "\ud53c\ubd80 \ud0c0\uc785\uc774\ub098 \uace0\ubbfc\uc5d0 \ub9de\ub294 \uc0c1\ud488\uc744 \ucd94\ucc9c\ud569\ub2c8\ub2e4. \uc608: '\uac74\uc131 \ud53c\ubd80\uc5d0 \uc88b\uc740 \ud06c\ub9bc \ucd94\ucc9c\ud574\uc918'"
    },
    "events": 

---
# SECTION 2: AFTER — Transparent Routing

**`AgentGroupChat` + `KernelFunctionSelectionStrategy`로 라우팅을 분리합니다.**

핵심 변화:
- **LLM call #1** (Selection): 어떤 agent를 선택할지 결정 (routing only)
- **`result_parser`** 가 선택 결과를 structured JSON으로 로깅
- **LLM call #2** (Response): 선택된 agent가 실제 응답 생성
- **Override hook**: 잘못된 라우팅을 커스텀 로직으로 교정 가능
- **OpenTelemetry** span으로 Application Insights에 trace 전송

| Routing Trace | Override | Logging Hook | OTel |
|:---:|:---:|:---:|:---:|
| ✅ agent, reason, confidence | ✅ Yes | ✅ JSON | ✅ App Insights |


In [6]:
# Cell 8: Section 2 — Transparent routing queries
from agents.after.transparent_routing import build_transparent_routing_chat

after_kernel = create_kernel()

queries_2 = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 2: Transparent Routing')
print('(AgentGroupChat + KernelFunctionSelectionStrategy)')
print('=' * 60)

for q in queries_2:
    print(f'\nQuery: {q}')
    chat, agents = build_transparent_routing_chat(
        kernel=after_kernel, logger=logger, tracer=tracer, query=q,
    )
    await chat.add_chat_message(message=q)
    async for response in chat.invoke():
        if response.content and response.name:
            print(f'\n[{response.name}]: {response.content[:500]}')
    print(f'\nRouting Trace: See structured JSON log above')
    print('-' * 60)


SECTION 2: Transparent Routing
(AgentGroupChat + KernelFunctionSelectionStrategy)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x42dfbef2eec4de4dba650224ca7e4f40",
        "span_id": "0xff46956c975c5e63",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-14T04:05:29.819312Z",
    "end_time": "2026-04-14T04:05:32.592657Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-14T04:05:32.5944

In [7]:
# Cell 9: Section 2 — Override demo with ambiguous query
ambiguous_query = '설화수 자음생 크림 건성 피부에 좋아?'

override_rules = {
    '설화수 자음생 크림 건성 피부에 좋아?': 'SearchAgent',
}

print('=' * 60)
print('SECTION 2: Override Demo — Ambiguous Query')
print('=' * 60)

# Without override
print(f'\n[Without Override]')
print(f'Query: {ambiguous_query}')
chat_no, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query,
)
await chat_no.add_chat_message(message=ambiguous_query)
async for resp in chat_no.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

# With override
print(f'\n[With Override] (override_applied: true)')
print(f'Query: {ambiguous_query}')
chat_ov, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query, override_rules=override_rules,
)
await chat_ov.add_chat_message(message=ambiguous_query)
async for resp in chat_ov.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

print(f'\nCompare logs: override_applied false -> true')
print('-' * 60)


SECTION 2: Override Demo — Ambiguous Query

[Without Override]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x1e5fcf7f4a6d57e3f60103cca047ea12",
        "span_id": "0xc775792c7384c587",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-14T04:05:53.414420Z",
    "end_time": "2026-04-14T04:05:54.817513Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-14T04:05:54.819099+00:00", "leve

---
# SECTION 3: AFTER+ — Self-Reflection

**Section 2 + explicit `reflect()` function으로 응답 품질을 검증합니다.**

Flow:
1. **Routing** → AgentGroupChat이 agent 선택 (Section 2와 동일)
2. **KB Retrieval** → 선택된 agent가 mock KB에서 데이터 조회
3. **Reflection** → `reflect()` 함수가 별도 LLM call로 응답 품질 평가
4. **Re-route (REPLACE)** → 품질 미달 시 다른 agent로 교체, 원래 응답 폐기

핵심: `reflect()`는 **SK framework hook에 의존하지 않는** 명시적 Python function  
→ SK / MAF / LangGraph 어디서든 동일하게 작동 (framework-portable)

| Routing | Reflection | Re-route | Framework Portable |
|:---:|:---:|:---:|:---:|
| ✅ Full | ✅ Explicit | ✅ REPLACE | ✅ SK/MAF/LangGraph |


In [9]:
# Cell 11: Section 3 — Setup
from agents.after.self_reflection_routing import reflect, ReflectionResult
from agents.after.transparent_routing import (
    build_transparent_routing_chat,
    RECOMMEND_AGENT, SEARCH_AGENT, POLICY_AGENT,
)

reflection_kernel = create_kernel()

print('Section 3 setup complete')
print('  reflect() is a plain Python async function')
print('  -> Zero SK framework hook dependency')
print('  -> Portable to MAF / LangGraph without change')
print('  Pattern: route -> respond -> reflect -> (re-route if needed) -> final')


Section 3 setup complete
  reflect() is a plain Python async function
  -> Zero SK framework hook dependency
  -> Portable to MAF / LangGraph without change
  Pattern: route -> respond -> reflect -> (re-route if needed) -> final


In [12]:
# Cell 12: Section 3 — Two cases: KEEP vs REPLACE
from semantic_kernel.contents import ChatHistory

async def run_reflection_cycle(query: str, case_label: str, override_rules: dict[str, str] | None = None):
    """Run the full reflection cycle for a given query."""
    print(f'\n{"=" * 60}')
    print(f'[{case_label}]')
    print(f'Query: {query}')
    print('=' * 60)

    with tracer.start_as_current_span(f'section3_{case_label}') as span:
        span.set_attribute('query', query)

        # Step 1: Route via AgentGroupChat
        chat, agents_map = build_transparent_routing_chat(
            kernel=reflection_kernel, logger=logger, tracer=tracer,
            query=query, override_rules=override_rules,
        )
        await chat.add_chat_message(message=query)

        first_agent_name = None
        first_response = None

        async for response in chat.invoke():
            if response.content and response.name:
                first_agent_name = response.name
                first_response = response.content
                print(f'\n[Step 1 - Routing] Selected: {first_agent_name}')
                print(f'[{first_agent_name}]: {first_response[:400]}')

        # Step 2: KB retrieval (mock)
        if first_agent_name == RECOMMEND_AGENT:
            kb_raw = rp.recommend_product(query)
        elif first_agent_name == SEARCH_AGENT:
            kb_raw = sp.search_product(query)
        else:
            kb_raw = pp.lookup_policy(query)

        kb_results = json.loads(kb_raw)
        kb_items = kb_results.get('recommendations', kb_results.get('results', kb_results.get('policies', [])))
        print(f'\n[Step 2 - KB Retrieval] {len(kb_items)} items from {first_agent_name}')

        # Step 3: Self-reflection (rule-based pre-check + LLM fallback)
        print(f'\n[Step 3 - Reflection] Evaluating response quality...')
        reflection = await reflect(
            query=query,
            agent_name=first_agent_name or '',
            agent_response=first_response or '',
            kb_results=kb_items,
            kernel=reflection_kernel,
        )

        print(f'  intent_match:    {reflection.intent_match}')
        print(f'  sufficient_info: {reflection.sufficient_info}')
        print(f'  should_reroute:  {reflection.should_reroute}')
        print(f'  reroute_to:      {reflection.reroute_to}')
        print(f'  reason:          {reflection.reason}')

        if 'PARSE_FAILED' in reflection.reason:
            print(f'\n⚠️ WARNING: Reflection LLM returned malformed JSON.')
            print(f'  Raw output: {reflection.reason}')
            print(f'  Keeping original response as fallback.')

        # Step 4: Re-route if needed (REPLACE pattern)
        if reflection.should_reroute and reflection.reroute_to:
            reroute_agent_name = reflection.reroute_to
            print(f'\n[Step 4 - REPLACE] Re-routing: {first_agent_name} -> {reroute_agent_name}')

            reroute_agent = agents_map.get(reroute_agent_name)
            if reroute_agent:
                reroute_history = ChatHistory()
                reroute_history.add_user_message(query)
                final_response = None
                async for msg in reroute_agent.invoke(reroute_history):
                    text = str(msg)
                    if text:
                        final_response = text
                        print(f'[Final - {reroute_agent_name}]: {final_response[:400]}')
                action = 'REPLACE'
                final_agent = reroute_agent_name
            else:
                print(f'Agent {reroute_agent_name} not found, keeping original')
                final_response = first_response
                action = 'KEEP'
                final_agent = first_agent_name
        else:
            print(f'\n[Step 4] Reflection passed — keeping original response')
            final_response = first_response
            action = 'KEEP'
            final_agent = first_agent_name

        # Log self-reflection event
        log_self_reflection(
            logger,
            query=query,
            agent=first_agent_name or '',
            kb_results_count=len(kb_items),
            reflection={
                'intent_match': reflection.intent_match,
                'sufficient_info': reflection.sufficient_info,
                'should_reroute': reflection.should_reroute,
                'reroute_to': reflection.reroute_to,
                'reason': reflection.reason,
            },
            action=action,
            final_agent=final_agent,
        )
        span.set_attribute('reflection.action', action)
        span.set_attribute('reflection.final_agent', final_agent or '')

    # Full decision trace
    print('\n' + '-' * 60)
    print(f'Full Decision Trace [{case_label}]:')
    print(f'  1. Routing    -> {first_agent_name}')
    print(f'  2. KB         -> {len(kb_items)} items retrieved')
    print(f'  3. Reflection -> should_reroute={reflection.should_reroute}')
    print(f'  4. Action     -> {action} (final agent: {final_agent})')
    print('-' * 60)

# ═══════════════════════════════════════════════════════════
# Case 1: KEEP — 추천 의도만 있는 쿼리 (비즈니스 규칙 미해당)
# ═══════════════════════════════════════════════════════════
await run_reflection_cycle(
    query='설화수 자음생 크림 건성 피부에 좋아?',
    case_label='Case 1: KEEP',
)

# ═══════════════════════════════════════════════════════════
# Case 2: REPLACE — override로 RecommendAgent 강제 → 비즈니스 규칙 트리거
# ═══════════════════════════════════════════════════════════
query_replace = '설화수 자음생 크림 가격이랑 건성 피부에 맞는지 알려줘'
await run_reflection_cycle(
    query=query_replace,
    case_label='Case 2: REPLACE',
    override_rules={query_replace: 'RecommendAgent'},  # 데모용: 의도적 미스라우팅
)



[Case 1: KEEP]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x89b7def7dce52852acb816ea8d993a35",
        "span_id": "0xe9cf83718ba7bd14",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2b86f5563f7ea15b",
    "start_time": "2026-04-14T05:20:35.081676Z",
    "end_time": "2026-04-14T05:20:38.593665Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-14T05:20:38.594878+00:00", "level": "INFO", "logger": "routing_

---
# SUMMARY

## Before vs After

| 항목 | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|------|:---:|:---:|:---:|:---:|
| **Routing 분리** | ❌ | ❌ | ✅ | ✅ |
| **Routing Trace** | ❌ | ❌ | ✅ agent, reason, confidence | ✅ + reflection |
| **Override Hook** | ❌ | ❌ | ✅ result_parser | ✅ |
| **Self-Reflection** | ❌ | ❌ | ❌ | ✅ explicit reflect() |
| **Re-route (REPLACE)** | ❌ | ❌ | ❌ | ✅ |
| **OTel Tracing** | ❌ | ❌ | ✅ | ✅ |
| **Framework Portable** | — | — | SK only | ✅ SK/MAF/LangGraph |

## Key Takeaways

1. **SK는 이미 투명한 라우팅을 공식 지원합니다** — `KernelFunctionSelectionStrategy`를 활용하면 기존 코드 구조를 크게 변경하지 않고도 라우팅 가시성을 확보할 수 있습니다.
2. **라우팅과 응답 생성을 분리하면 운영이 쉬워집니다** — 로깅, 오버라이드, 모니터링이 자연스럽게 따라옵니다.
3. **Self-reflection은 framework에 종속되지 않습니다** — explicit function으로 구현하면 어디로든 이식 가능합니다.


In [11]:
# Cell 14: Summary — comparison table + App Insights KQL
from IPython.display import display, Markdown

comparison = """
## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|---------|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + respond + reflect [+ re-route]) |
| Routing Visible | No | No | Yes | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser |
| Quality Check | No | No | No | reflect() |
| Re-route on Failure | No | No | No | REPLACE |
| OTel / App Insights | No | No | Yes | Yes |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() |
"""
display(Markdown(comparison))

kql = """
## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```
"""
display(Markdown(kql))

print('\nDemo complete!')



## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|---------|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + respond + reflect [+ re-route]) |
| Routing Visible | No | No | Yes | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser |
| Quality Check | No | No | No | reflect() |
| Re-route on Failure | No | No | No | REPLACE |
| OTel / App Insights | No | No | Yes | Yes |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() |



## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```



Demo complete!


In [ ]:
#